# **msProf工具使用技巧**

本节为msProf工具使用技巧介绍章节，完成本章节内容的学习可以掌握如何使用profiling工具分析算子性能瓶颈。我们将按照以下结构，带你学习msProf工具使用：
- 环境准备
- 什么是msProf工具
- 如何使用msProf工具采集上板性能数据
- 如何分析性能数据
- 针对性优化
- 课后实践

---

## **1 环境准备**
本文所有内容均存放于Sources文件夹。
在开始创建算子工程前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，并完成代码目录的创建。保证能正常导入代码以及正常使用工具。

In [ ]:
!mkdir -p Sources/08.02

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

---
## **2 什么是msProf工具**
msProf工具用于采集和分析运行在昇腾AI处理器上算子的关键性能指标，用户可根据输出的性能数据，快速定位算子的软、硬件性能瓶颈，提升算子性能的分析效率。
当前支持基于不同运行模式（上板或仿真）和不同文件形式（可执行文件或算子二进制.o文件）进行性能数据的采集和自动解析。  
**msProf工具的使用依赖CANN包中的msopprof可执行文件，该文件中的接口使用和msprof op一致，该文件为CANN包自带，无需单独安装。**

---
## **3 如何使用msProf工具采集上板性能数据**
本节课讲演示如何使用msProf工具上板模式抓取单算子API调用程序执行时算子的各个性能数据，在使用工具前，我们需要先了解一些关键参数的含义。详细参数参考工具概述的[命令汇总](https://www.hiascend.com/document/redirect/CANNCommunitymsOpProf)介绍。

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 1485px; border: 1px solid #ddd;">
  <tr>
    <td style="width: 30%; padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">参数名</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">参数作用</td>
  </tr>
  <tr>
    <td style="width: 30%; padding: 8px; border-bottom: 1px solid #ddd;"><code>--kernel-name</code></td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">指定要采集的算子名称，支持使用算子名前缀进行模糊匹配。如果不指定，则只对程序运行过程中调度的第一个算子进行采集。</td>
  </tr>
  <tr>
    <td style="width: 30%; padding: 8px; border-bottom: 1px solid #ddd;"><code>--launch-count</code></td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">设置可以采集算子的最大数量，默认值为1，取值范围为1~5000之间的整数。</td>
  </tr>
  <tr>
    <td style="width: 30%; padding: 8px; border-bottom: 1px solid #ddd;"><code>--launch-skip-before-match</code></td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">用于设置不需要采集数据的算子数量，从第一个算子开始到指定数目的算子不进行采集，仅对指定数目之后的算子开始采集。</td>
  </tr>
  <tr>
    <td style="width: 30%; padding: 8px; border-bottom: 1px solid #ddd;"><code>--warm-up</code></td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">部分算子使用 <code>msprof op</code> 采集时，会达不到芯片提频的最小任务耗时产生降频，从而会对交付件的结果产生一定影响。在该情况下，可用 <code>--warm-up</code> 指定预热次数，提前提升AI处理器的运行频率，使上板数据更准确。</td>
  </tr>
  <tr>
    <td style="width: 30%; padding: 8px;"><code>--output</code></td>
    <td style="padding: 8px;">收集到的性能数据的存放路径，默认在当前目录下保存性能数据。</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

### **3.1 准备调用程序**
在抓取性能前，我们需要先部署好算子，并完成单算子调用测试，确保算子可以被正常调用，这里我们使用之前课程的AddCustomTemplate算子和其调用程序。
算子工程目录结构为：
```
custom_op/
├── framework/
├── op_host/
│   ├── add_custom_template.cpp          // 包含算子原型注册、tiling实现 shape与Dtype推导内容
│   └── CMakeLists.txt                   // host侧CMakeLists文件，一般不需要修改
├── op_kernel/
│   ├── add_custom_template_tiling.h     // 算子tiling定义文件   
│   ├── add_custom_template.cpp          // 算子代码实现文件 
│   └── CMakeLists.txt                   // kernel侧CMakeLists文件，一般不需要修改
├── CMakeLists.txt                       // 根目录CMakeLists文件，一般不需要修改
├── CMakePresets.json                    // CMake编译配置文件
└── build.sh                             // 编译脚本
```  

执行以下代码复制已准备好的AddCustomTemplate算子工程和测试程序，测试输入为[8, 2048]的half类型数据。

In [ ]:
# 清理已存在的custom_op
!rm -rf Sources/08.02/custom_op

# 复制之前课程的算子工程, 并修改算子工程内CANN包路径配置为实际路径
!cp -r src/custom_op Sources/08.02/custom_op;sed -i '/"ASCEND_CANN_PACKAGE_PATH": {/,/}/ s|\s*"value": ".*"| "value": "'"$ASCEND_TOOLKIT_HOME"'"|' Sources/08.02/custom_op/CMakePresets.json

# 编译部署算子
!cd Sources/08.02/custom_op;bash build.sh;./build_out/custom_opp*.run  --install-path=${HOME}/

# 清除已有的测试代码
!rm -rf Sources/08.02/test;

# 复制已有的测试代码
!cp -r src/custom_op/test Sources/08.02/test;

# 编译调用代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/08.02/test/main.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/08.02/execute_op;

# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./Sources/08.02/execute_op

执行成功后会有如下输出，说明算子可以正常调用:
```
result is:
3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0
test pass
```

### **3.2 使用msProf工具采集性能数据**
使用msProf工具的上板采集模式对已编译好的调用程序进行性能采集，采集命令为:
```shell
msprof op --output=Sources/08.02/prof ./execute_op
```
* **--output**: 指定抓取的性能数据存放路径  

* **./execute_op**: 已编译好的测试程序，**如果可执行程序有参数，则该参数必须放命令最后，可执行程序前的参数为msProf工具参数，可执行程序后的参数为可执行程序的参数**

In [ ]:
# 清除可能存在的性能文件
!rm -rf Sources/08.02/prof
# 创建性能文件存放目录
!mkdir -p Sources/08.02/prof
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;msprof op --output=Sources/08.02/prof ./Sources/08.02/execute_op

命令执行后，Sources/08.02/prof目录下会生成性能数据文件，目录结构如下:

```
OPPROF_20260309094753_QQXMCEXGHREUCDKS/
├── visualize_data.bin                     // 算子基础信息、计算单元负载、热点函数和Roofline瓶颈分析等信息的可视化呈现文件
├── OpBasicInfo.csv                        // 算子基础信息，包含算子名称、block dim和耗时等信息
├── ResourceConflictRatio.csv              // UB上的bank group、bank conflict和资源冲突在所有指令中的占比
├── PipeUtilization.csv                    // 采集计算单元和搬运单元耗时和占比
├── Memory.csv                             // UB/L1/L2/主存储器采集内存读写带宽速率
├── L2Cache.csv                            // L2 Cache命中率
├── MemoryL0.csv                           // L0A/L0B/L0C采集内存读写带宽速率
├── dump/                                  // 原始的性能数据文件夹，用户无需关注
├── ArithmeticUtilization.csv              // Cube和Vector类型的指令耗时和占比
└── MemoryUB.csv                           // mte/vector/scalar采集ub读写带宽速率
```


执行以下命令查看Sources/08.02/prof查看到采集的性能数据文件。

In [ ]:
!cd Sources/08.02/prof; find . -maxdepth 2 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

---
## **4 如何分析性能数据**

理论性能为算子实际性能的理想目标。不同的硬件平台的硬件规格各异，理论性能可以帮助我们了解硬件的潜能，从而设定性能优化的目标。

### **4.1 搬运相关流水理论耗时（MTE1/MTE2/MTE3等）**
**计算公式：**
搬运相关流水的理论耗时 = 搬运数据量（单位：Byte） / 理论带宽

**示例：**
某款AI处理器的GM峰值带宽约为1.8TB/s，想要进行一次float数据类型、4096 * 4096大小的矩阵搬运，搬运的理论耗时是：
sizeof(float) * 4096 * 4096 / 1.8TB/s = 37.28us（按照1TB = 10^12 Byte来计算）

**说明：**
- 搬运指令同时存在时，会存在共享带宽的情况，并不能每条都以接近理论带宽的速率搬运数据。  
- 搬运不同大小的数据块时，对带宽的利用率（有效带宽/理论带宽）不一样。针对每次搬运数据量较小的情况，实测性能达不到理论带宽。

### **4.2 计算相关流水理论耗时（Cube/Vector/Scalar等）**
**计算公式：**
计算相关流水的理论耗时 = 计算数据量（单位：Element） / 理论算力

**示例：**
某款AI处理器对float数据类型的Vector理论峰值算力为11.06TOPS，想要进行一次32K float类型的Element单指令计算，计算的理论耗时是：
32K / 11.06TOPS = 0.003us（按照1K = 1000来计算）

### **4.3 查找瓶颈**
获取性能数据后，**耗时较长的流程**、**实际数值与理论数值差异较大的环节**，均被判定为性能“瓶颈点”。我们需要基于抓取的性能数据，重点分析哪些环节属于“耗时较长的流程”。

分析性能数据需先掌握性能数据的基础查看方法，本示例以 Vector 算子为例展开，因此暂不展示 Cube 相关解释。
> 完整参数说明可参考：[PipeUtilization 字段解释](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition/900beta1/devaids/optool/atlasopdev_16_0099.html)


<table border="1" cellpadding="8" cellspacing="0" width="100%">
  <thead>
    <tr bgcolor="#f5f5f5">
      <th style="width: 25%;">字段名称</th>
      <th style="width: 75%;">字段说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>block_id</td>
      <td>Task运行切分数量，对应Task运行时配置的核数。</td>
    </tr>
    <tr>
      <td>sub_block_id</td>
      <td>Task运行使用的每个block名称和序号。</td>
    </tr>
    <tr>
      <td>aiv_time(us)</td>
      <td>该Task被分配到每个AI Vector Core计算单元上后，每个AI Vector Core计算单元上的执行时间，单位us。</td>
    </tr>
    <tr>
      <td>aiv_total_cycles</td>
      <td>该Task被分配到每个AI Vector Core计算单元上后，每个AI Vector Core计算单元上的执行的cycle总数。</td>
    </tr>
    <tr>
      <td>aiv_vec_time(us)</td>
      <td>代表vec类型指令（向量类运算指令）耗时。</td>
    </tr>
    <tr>
      <td>aiv_vec_ratio</td>
      <td>代表vec类型指令（向量类运算指令）的cycle数在total cycle数中的占用比。</td>
    </tr>
    <tr>
      <td>aiv_scalar_time(us)</td>
      <td>代表scalar类型指令（标量类运算指令）耗时。</td>
    </tr>
    <tr>
      <td>aiv_scalar_ratio</td>
      <td>代表scalar类型指令（标量类运算指令）的cycle数在total cycle数中的占用比。</td>
    </tr>
    <tr>
      <td>aiv_mte2_time(us)</td>
      <td>代表MTE2类型指令（GM->AICORE搬运类指令）耗时。</td>
    </tr>
    <tr>
      <td>aiv_mte2_ratio</td>
      <td>代表MTE2类型指令（GM->AICORE搬运类指令）的cycle数在total cycle数中的占用比。</td>
    </tr>
    <tr>
      <td>aiv_mte3_time(us)</td>
      <td>代表MTE3类型指令（AICORE->GM搬运类指令）耗时。</td>
    </tr>
    <tr>
      <td>aiv_mte3_ratio</td>
      <td>代表MTE3类型指令（AICORE->GM搬运类指令）的cycle数在total cycle数中的占用比。</td>
    </tr>
    <tr>
      <td>aiv_icache_miss_rate</td>
      <td>代表ICache缺失率，即未命中instruction的L1 cache，数值越小越好。</td>
    </tr>
    <tr>
      <td>aiv_mte3_active_bw(GB/s)</td>
      <td>代表MTE3类型指令（AICORE->DDR AIV搬运类指令）数据量对应active cycle的活跃带宽。</td>
    </tr>
    <tr>
      <td>aiv_mte2_active_bw(GB/s)</td>
      <td>代表MTE2类型指令（DDR->AICORE AIV搬运类指令）数据量对应active cycle的活跃带宽。</td>
    </tr>
  </tbody>
</table>

#### **4.4 耗时较长的流程分析**
对照上表，执行以下命令查看我们抓取到的性能数据表格，分析搬运、计算、阻塞等耗时，找出耗时较长的流程。

In [ ]:
import pandas as pd
import glob

csv_file = glob.glob("Sources/08.02/prof/*/PipeUtilization.csv")[0]
df = pd.read_csv(csv_file)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df

示例的代码没有开启DOUBLE BUFFER，所以在profiling数据中，所有耗时为搬入耗时 + 运算耗时 + 搬出耗时。所以我们需要关注aiv_time(us)、aiv_scalar_time(us)、aiv_mte2_time(us)、aiv_vec_time(us)、aiv_mte3_time(us)的耗时。从性能数据表格中，我们可以看到scalar占比超过了90%，那么scalar运算耗时为重点优化对象。  

<img src="./images/printf.png" alt="prof"  width="700px">

---
## **5 针对性优化**
分析完性能数据后，根据数据结果分析出的较长耗时流程，针对性优化。以示例代码的scalar运算耗时较长为例，我们应该分析代码，看代码中是否有scalar运算，哪部分scalar运算可能会导致耗时较长，找到对应代码后我们可以考虑是否可以将scalar运算放到host侧进行，以减少scalar运算的耗时。查看Kernel侧实现，我们可以看到，scalar运算有三部分内容，分别为：
1. 第一部分为Init函数内对tileLength的长度计算和对x、y、z的GlobalMemory偏移计算。
2. 第二部分为Process函数的循环控制和printf打印。
3. 第三部分为CopyIn、CopyOut函数内的地址偏移计算。

分析这三部分，我们可以发现，只有第一部分的tileLength计算可以放到host侧进行，第二部分的printf打印可以屏蔽。其他部分代码需要保持原来的实现，不能放到host侧进行。而tilingLength计算量较小，不会对性能造成较大影响，所以推测将printf打印屏蔽后，算子的性能会有较大提升。尝试根据分析修改以下代码：

In [ ]:
%%writefile Sources/08.02/custom_op/op_kernel/add_custom_template.cpp 
#include "kernel_operator.h"
#include "add_custom_template_tiling.h"
#include "kernel_operator_dump_tensor_intf_impl.h"
constexpr int32_t BUFFER_NUM = 1;

template <class dtypeX, class dtypeY, class dtypeZ>
class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
    {
        this->blockLength = totalLength / AscendC::GetBlockNum();
        this->tileNum = tileNum;
        this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
        xGm.SetGlobalBuffer((__gm__ dtypeX *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        yGm.SetGlobalBuffer((__gm__ dtypeY *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        zGm.SetGlobalBuffer((__gm__ dtypeZ *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(dtypeX));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(dtypeY));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(dtypeZ));
    }

    __aicore__ inline void Process()
    {
        int32_t loopCount = this->tileNum * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
        AscendC::printf("Core %d executed %d times in total\n",  AscendC::GetBlockIdx(), loopCount);
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<dtypeX> xLocal = inQueueX.AllocTensor<dtypeX>();
        AscendC::LocalTensor<dtypeY> yLocal = inQueueY.AllocTensor<dtypeY>();
        AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
        AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<dtypeX> xLocal = inQueueX.DeQue<dtypeX>();
        AscendC::LocalTensor<dtypeY> yLocal = inQueueY.DeQue<dtypeY>();
        AscendC::LocalTensor<dtypeZ> zLocal = outQueueZ.AllocTensor<dtypeZ>();
        AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);
        outQueueZ.EnQue<dtypeZ>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<dtypeZ> zLocal = outQueueZ.DeQue<dtypeZ>();
        AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<dtypeX> xGm;
    AscendC::GlobalTensor<dtypeY> yGm;
    AscendC::GlobalTensor<dtypeZ> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

 __global__ __aicore__ void add_custom_template(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling)
{
    REGISTER_TILING_DEFAULT(TilingDataTemplate);
    GET_TILING_DATA_WITH_STRUCT(TilingDataTemplate, tiling_data, tiling);
    KernelAdd<DTYPE_X, DTYPE_Y, DTYPE_Z> op;
    op.Init(x, y, z, tiling_data.totalLength, tiling_data.tileNum);
    op.Process();
}

修改完成后重新部署算子并抓取性能：

In [ ]:
# 编译部署算子
!cd Sources/08.02/custom_op;bash build.sh;./build_out/custom_opp*.run  --install-path=${HOME}/

# 清除可能存在的性能文件
!rm -rf Sources/08.02/prof2
# 创建性能文件存放目录
!mkdir -p Sources/08.02/prof2
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;msprof op --output=Sources/08.02/prof2 ./Sources/08.02/execute_op

抓取性能后，读取性能数据与之前对比，观察是否符合预期，减少了scalar运算耗时。

In [ ]:
import pandas as pd
import glob

csv_file = glob.glob("Sources/08.02/prof2/*/PipeUtilization.csv")[0]
df = pd.read_csv(csv_file)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df

预期scalar耗时大幅度降低：  

<img src="./images/no_printf.png" alt="prof_no_printf"  width="700px">

---
## **课后实践**
请根据本节课程学习内容完成以下题目进行自测。

1. （单选题）msProf工具的核心作用是（）  
    A. 自动编写算子核心代码  
    B. 采集并分析算子运行时的各类性能统计数据  
    C. 仅验证算子功能是否正确  
    D. 替代 CMake 完成算子编译构建  

2. （单选题） 使用msProf工具分析算子性能时，其统计的性能数据不包含以下哪类（）  
    A. 算子执行过程中的整体耗时  
    B. 计算单元与数据搬运单元的耗时占比  
    C. 算子代码的语法错误信息  
    D. 不同存储层级的内存读写相关指标  

3. （单选题）关于msProf工具的性能统计维度，以下说法正确的是（）  
    A. 仅能统计算子的整体执行耗时，无细分指标  
    B. 可对硬件计算单元的指令执行情况做量化统计  
    C. 无法统计存储层级的资源冲突相关占比  
    D. 仅支持离线查看算子性能，不支持实时统计  

4. （单选题）利用msProf工具的性能统计结果，主要能为算子优化提供什么核心依据（）  
    A. 定位算子性能瓶颈所在的环节  
    B. 自动生成优化后的算子代码  
    C. 修复算子的功能逻辑错误  
    D. 调整算子的编译环境配置  

5. （单选题）以下哪项是使用msProf工具做算子性能分析的前提条件（）  
    A. 算子已完成开发并能正常运行  
    B. 仅需编写好算子的头文件，无需实现核心逻辑  
    C. 无需配置Ascend C开发环境  
    D. 算子能在GPU上运行  

执行以下代码查看答案：

In [ ]:
!cat ./answer/08.02_answer.txt